# SAGE Public URL Notebook

Use this notebook to:

1. install the ngrok dependency
2. set your ngrok auth token
3. start the local SAGE FastAPI server
4. expose it on a public URL

The server endpoint you will expose is:

- `GET /health`
- `POST /generate`


In [ ]:
%pip install -q pyngrok uvicorn fastapi httpx

In [ ]:
import os
from pathlib import Path

# Paste your ngrok token here, or leave this as-is and set NGROK_AUTHTOKEN in the environment.
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

REPO_ROOT = Path.cwd()
HOST = "127.0.0.1"
PORT = 8000

if NGROK_AUTHTOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE":
    os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN

print(f"Repo root: {REPO_ROOT}")
print(f"Server target: http://{HOST}:{PORT}")

In [ ]:
import atexit
import subprocess
import sys
import time

import httpx

SERVER_PROCESS = globals().get("SERVER_PROCESS")

def stop_server():
    global SERVER_PROCESS
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
        print("Stopped existing server process.")
    SERVER_PROCESS = None

def wait_for_server(url: str, timeout_seconds: int = 30):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            response = httpx.get(url, timeout=2.0)
            if response.status_code == 200:
                return response.json()
        except Exception as exc:
            last_error = exc
        time.sleep(1)
    raise RuntimeError(f"Server did not become ready in time. Last error: {last_error}")

stop_server()
SERVER_PROCESS = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "serve.server:app", "--host", HOST, "--port", str(PORT)],
    cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
atexit.register(stop_server)

health = wait_for_server(f"http://{HOST}:{PORT}/health")
print("Local server is ready:")
print(health)

In [ ]:
from pyngrok import ngrok

NGROK_TUNNEL = globals().get("NGROK_TUNNEL")

def stop_tunnel():
    global NGROK_TUNNEL
    if NGROK_TUNNEL is not None:
        ngrok.disconnect(NGROK_TUNNEL.public_url)
        NGROK_TUNNEL = None
        print("Closed existing ngrok tunnel.")

auth_token = os.environ.get("NGROK_AUTHTOKEN")
if not auth_token:
    raise RuntimeError("Set NGROK_AUTHTOKEN first. Paste your token in the config cell above.")

stop_tunnel()
ngrok.set_auth_token(auth_token)
NGROK_TUNNEL = ngrok.connect(addr=PORT, proto="http", bind_tls=True)
atexit.register(stop_tunnel)

print("Public URL:", NGROK_TUNNEL.public_url)
print("Health URL:", f"{NGROK_TUNNEL.public_url}/health")
print("Generate URL:", f"{NGROK_TUNNEL.public_url}/generate")

In [ ]:
public_health = httpx.get(f"{NGROK_TUNNEL.public_url}/health", timeout=15.0)
print("Public health status:", public_health.status_code)
print(public_health.json())

In [ ]:
# Example low-level generate call.
# Replace input_ids with real tokenizer output after you have a trained tokenizer and model checkpoint.
payload = {"input_ids": [1, 42, 99], "max_new_tokens": 8}
response = httpx.post(f"{NGROK_TUNNEL.public_url}/generate", json=payload, timeout=30.0)
print(response.status_code)
print(response.json())

In [ ]:
# Optional cleanup cell
stop_tunnel()
stop_server()